# Parameter Fitting for Reaction Engineering

This notebook introduces parameter estimation techniques used in chemical reaction engineering.

Experimental data rarely provide reaction parameters directly.

Engineers often need to estimate:
- reaction rate constants
- reaction orders
- activation energies
- kinetic model parameters

The goal of this notebook is to connect experimental data with mathematical models.

## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from scipy.optimize import curve_fit

## 1. Why parameter fitting matters

Reaction engineering models usually contain unknown parameters.

For example, there is a first-order reaction:

$$
-r_A=kC_A
$$


The reaction rate constant $k$ must be obtained from experiments.


Parameter fitting finds the values that make the model best describe the data.

## 2. Generating synthetic reaction data

Assume experimental measurements are collected from a batch reactor.

For a first-order reaction:

$$
C_A=C_{A0}e^{-kt}
$$

Experimental data usually contain measurement noise.

In [ ]:
np.random.seed(0)

k_true = 0.15
CA0 = 1.0

time = np.linspace(0,30,20)

CA_true = CA0*np.exp(-k_true*time)

noise = np.random.normal(
    0,
    0.02,
    size=len(time)
)

CA_exp = CA_true + noise

In [ ]:
plt.figure(figsize=(8,4))

plt.scatter(
    time,
    CA_exp,
    label="Experimental data"
)

plt.plot(
    time,
    CA_true,
    label="True model"
)

plt.xlabel("Time")
plt.ylabel("CA")

plt.legend()
plt.grid(True)

plt.show()

## 3. Linear fitting with NumPy

For a first-order reaction:

$$
\ln(C_A)=\ln(C_{A0})-kt
$$

This equation can be transformed into a linear form:

$$
y=ax+b
$$

where:

$$
y=\ln(C_A)
$$

$$
x=t
$$

The slope gives the rate constant.

In [ ]:
ln_CA = np.log(CA_exp)

slope, intercept = np.polyfit(
    time,
    ln_CA,
    1
)

k_estimated = -slope

k_estimated

In [ ]:
plt.figure(figsize=(8,4))

plt.scatter(
    time,
    ln_CA
)

plt.plot(
    time,
    slope*time+intercept
)

plt.xlabel("Time")
plt.ylabel("ln(CA)")

plt.grid(True)

plt.show()

## 4. Nonlinear fitting with SciPy curve_fit

Linearization is convenient but changes the error structure.

A more general approach is nonlinear regression.

SciPy provides:
curve_fit()
which directly fits the original equation.

In [ ]:
def first_order_model(t,k):

    return CA0*np.exp(-k*t)

In [ ]:
parameter, covariance = curve_fit(
    first_order_model,
    time,
    CA_exp,
    p0=[0.1]
)

k_fit = parameter[0]

k_fit

In [ ]:
CA_fit = first_order_model(
    time,
    k_fit
)

plt.figure(figsize=(8,4))

plt.scatter(
    time,
    CA_exp,
    label="Experiment"
)

plt.plot(
    time,
    CA_fit,
    label="Fitted model"
)

plt.xlabel("Time")
plt.ylabel("CA")

plt.legend()
plt.grid(True)

plt.show()

## 5. Determining reaction order

A general rate law:

$$
-r_A=kC_A^n
$$

where $n$ is the reaction order.

Experimental data can be used to estimate $n$.

In [ ]:
# Generate data

CA = np.linspace(
    0.1,
    1,
    20
)

n_true = 2
k_true = 0.5

rate = k_true*CA**n_true

rate_noise = rate + np.random.normal(
    0,
    0.01,
    len(rate)
)

In [ ]:
# log fitting

log_CA=np.log(CA)

log_rate=np.log(rate_noise)

n_est, log_k = np.polyfit(
    log_CA,
    log_rate,
    1
)

n_est

In [ ]:
plt.figure(figsize=(8,4))

plt.scatter(
    log_CA,
    log_rate
)

plt.plot(
    log_CA,
    n_est*log_CA+log_k
)

plt.xlabel("ln(CA)")
plt.ylabel("ln(rate)")

plt.grid(True)

plt.show()

## 6. Arrhenius parameter estimation

The Arrhenius equation:

$$
k=Ae^{-E_a/RT}
$$

Taking logarithm:

$$
\ln(k)=\ln(A)-\frac{E_a}{R}\frac{1}{T}
$$

The slope gives activation energy.

In [ ]:
R=8.314

T=np.array([
    300,
    320,
    340,
    360,
    380
])

Ea_true=50000

A_true=1e6

k=A_true*np.exp(
    -Ea_true/(R*T)
)

In [ ]:
x=1/T
y=np.log(k)

slope, intercept=np.polyfit(
    x,
    y,
    1
)

Ea_est=-slope*R

Ea_est

In [ ]:
plt.figure(figsize=(8,4))

plt.scatter(
    x,
    y
)

plt.plot(
    x,
    slope*x+intercept
)

plt.xlabel("1/T")
plt.ylabel("ln(k)")

plt.grid(True)

plt.show()

## 7. Model comparison

Different kinetic models can be compared using:
- visual agreement
- residual analysis
- fitting error

A good model should describe experimental behavior without unnecessary complexity.

In [ ]:
residuals = CA_exp - CA_fit

plt.figure(figsize=(8,4))

plt.scatter(
    time,
    residuals
)

plt.axhline(0)

plt.xlabel("Time")
plt.ylabel("Residual")

plt.grid(True)

plt.show()

## 8. Engineering interpretation

Parameter fitting connects experiments and models.

The same approach is used for:

- reactor design
- catalyst kinetics
- enzyme kinetics
- fermentation models
- transport models

Reliable parameter estimation is essential for predictive simulation.

## 9. Summary

In this notebook, you learned:

- why kinetic parameters need to be estimated
- linear regression for kinetic models
- nonlinear fitting using curve_fit
- reaction order estimation
- Arrhenius parameter estimation

These methods will be used throughout reaction engineering analysis.